# Phase 2: Xenium 5k predictability atlas: embeddings + patch-aligned pseudo-spots (+ Phase 4 Visium)

**Runtime:** GPU **L4, High-RAM ON** (TENX160 has ~1-2 B transcripts; the cell x gene matrix needs ~10-20 GB RAM).
A100 also works (faster GPU part, but the transcript streaming is CPU/network-bound, so L4 is the better value).
Expected ~2-3 h on L4 (~4-6 compute units at ~1.5 CU/h). Every step is resumable: if the runtime disconnects, just
re-run all cells and finished work is skipped.

**Secrets:** `HF_TOKEN` in Colab Secrets (never printed or saved).

**Section A (Xenium 5k, 6 samples, one at a time):**
1. download HEST `patches/<id>.h5` (224 px at 0.5 um/px, 112 um footprint), embed with UNI2-h, Virchow2, CONCH v1.5,
   H-optimus-1 (only if access works; otherwise skipped and recorded) and UNI2-h on the central 56 um
   (`uni2_h_c56`, patch-size sensitivity); delete the patch file;
2. download `transcripts/<id>_transcripts.parquet` to the VM disk (deleted right after), read it row group by row group,
   build the cell x gene matrix (qv >= 20, real genes, assigned cells), cell centroids in H&E pixels, coarse
   marker-based cell types, and pseudo-spot counts = sum of cells whose centroid lies in the patch footprint
   (window scale 1.0) plus 0.5x and 2x windows for Phase 5;
3. download HEST `st/<id>.h5ad` only to cross-check our pseudo-spots against HEST's (QC numbers saved), then delete.

TENX190 is skipped on purpose: HEST labels it Xenium 5k but it is the Xenium v1 (541-gene) run.

**Section B (Phase 4, 13 Visium samples of lung / ovary / cervix):** embeddings with the same encoders
and expression restricted to the Xenium 5k genes.

**Output (Drive):** `MyDrive/hest_fm_project/embeddings/phase2/xenium5k/...` and `.../embeddings/phase4/visium/...`

**Afterwards download into `data/embeddings/` (keeping the folder structure):**
- `phase2/xenium5k/{uni2_h,virchow2,conch_v15,hoptimus1}/`, `expression/`, `cells/`, `qc/`  (~3 GB)
- `phase4/visium/` (~0.5 GB)
- optional for Phase 5: `phase2/xenium5k/{uni2_h_c56,expression_w050,expression_w200}/` (~2 GB)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys
os.makedirs('/content/lib', exist_ok=True)
sys.path.insert(0, '/content/lib')

In [ ]:
!pip -q install 'timm>=1.0.9' 'huggingface_hub>=0.24' h5py anndata pyarrow einops einops-exts 'transformers==4.46.3'

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from huggingface_hub import login
# Token comes from Colab Secrets (key icon on the left -> HF_TOKEN, notebook access ON). Never printed.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
from huggingface_hub import whoami
print('HF login ok:', bool(whoami().get('name')))

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/hest_fm_project'
OUT = f'{DRIVE_ROOT}/embeddings/phase2/xenium5k'
OUT_VIS = f'{DRIVE_ROOT}/embeddings/phase4/visium'
SCRATCH = '/content/scratch'
SAMPLES = ['TENX189', 'TENX188', 'TENX187', 'NCBI887', 'NCBI888', 'TENX160']
VISIUM = ['TENX62', 'TENX72', 'MISC13', 'MISC14', 'MISC15', 'MISC16', 'MEND85', 'MEND86', 'MEND89', 'MEND90', 'TENX65', 'TENX51', 'TENX50']
ENCODERS = ['uni2_h', 'virchow2', 'conch_v15', 'hoptimus1', 'uni2_h_c56']
VISIUM_ENCODERS = ['uni2_h', 'virchow2', 'conch_v15', 'hoptimus1']
WINDOWS = {'expression': 1.0, 'expression_w050': 0.5, 'expression_w200': 2.0}
MIN_QV = 20
HEST = 'MahmoodLab/hest'
for d in (OUT, OUT_VIS, SCRATCH):
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile /content/lib/encoders.py
"""Frozen patch encoders for Colab extraction. Inlined into notebooks/colab_*.ipynb by scripts/build_notebooks.py.

Preprocessing runs on the GPU from uint8 HWC patches (224 px at 0.5 um/px from HEST patch files):
resize (only if the encoder's input size differs) -> /255 -> normalise.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable

import numpy as np
import torch
import torch.nn.functional as F

IMAGENET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
HOPTIMUS = ((0.707223, 0.578729, 0.703617), (0.211883, 0.230117, 0.177517))


@dataclass
class Encoder:
    name: str
    model: torch.nn.Module
    forward: Callable[[torch.Tensor], torch.Tensor]
    input_size: int
    mean: tuple
    std: tuple
    interpolation: str = "bilinear"
    amp: bool = True
    micro_batch: int = 256
    dim: int | None = None
    device: str = "cuda"
    center_crop_px: int | None = None   # crop the centre of the input patch first (smaller field of view)


def preprocess(batch_uint8: np.ndarray | torch.Tensor, enc: Encoder, device="cuda") -> torch.Tensor:
    x = torch.as_tensor(batch_uint8).to(device, non_blocking=True)
    x = x.permute(0, 3, 1, 2).float().div_(255.0)
    if enc.center_crop_px:
        o = (x.shape[-1] - enc.center_crop_px) // 2
        x = x[..., o:o + enc.center_crop_px, o:o + enc.center_crop_px]
    if x.shape[-1] != enc.input_size or x.shape[-2] != enc.input_size:
        x = F.interpolate(x, size=(enc.input_size, enc.input_size), mode=enc.interpolation,
                          align_corners=False, antialias=enc.input_size < x.shape[-1])
    mean = torch.tensor(enc.mean, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(enc.std, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


@torch.inference_mode()
def embed(batch_uint8, enc: Encoder, device=None, amp_dtype=torch.bfloat16) -> np.ndarray:
    device = device or enc.device
    outs = []
    for i in range(0, len(batch_uint8), enc.micro_batch):
        x = preprocess(batch_uint8[i:i + enc.micro_batch], enc, device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=enc.amp and device == "cuda"):
            z = enc.forward(x)
        outs.append(z.float().cpu().numpy())
    return np.concatenate(outs).astype(np.float16)


def build_encoder(name: str, device="cuda") -> Encoder:
    import timm

    if name == "resnet50":
        # HEST-Benchmark "resnet50": ImageNet ResNet50 truncated after layer3, global average pool -> 1024-d.
        m = timm.create_model("resnet50.tv_in1k", pretrained=True, features_only=True, out_indices=[3])
        fwd = lambda x: m(x)[0].mean(dim=(2, 3))
        enc = Encoder(name, m, fwd, 224, *IMAGENET, amp=False, micro_batch=256, dim=1024)
    elif name == "uni2_h":
        kw = dict(img_size=224, patch_size=14, depth=24, num_heads=24, init_values=1e-5, embed_dim=1536,
                  mlp_ratio=2.66667 * 2, num_classes=0, no_embed_class=True, mlp_layer=timm.layers.SwiGLUPacked,
                  act_layer=torch.nn.SiLU, reg_tokens=8, dynamic_img_size=True)
        m = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **kw)
        enc = Encoder(name, m, m, 224, *IMAGENET, micro_batch=256, dim=1536)
    elif name == "virchow2":
        m = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True,
                              mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU)

        def fwd(x):
            t = m(x)  # (B, 1 + 4 reg + 256, 1280)
            return torch.cat([t[:, 0], t[:, 5:].mean(1)], dim=-1)
        enc = Encoder(name, m, fwd, 224, *IMAGENET, interpolation="bicubic", micro_batch=256, dim=2560)
    elif name == "conch_v15":
        from transformers import AutoModel
        titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
        m, _ = titan.return_conch()   # eval transform: bilinear resize 448, ImageNet norm
        del titan
        enc = Encoder(name, m, m, 448, *IMAGENET, interpolation="bilinear", micro_batch=128, dim=768)
    elif name == "hoptimus1":
        m = timm.create_model("hf-hub:bioptimus/H-optimus-1", pretrained=True, init_values=1e-5,
                              dynamic_img_size=False)
        enc = Encoder(name, m, m, 224, *HOPTIMUS, micro_batch=256, dim=1536)
    elif name.startswith("uni2_h_c"):
        # UNI2-h on the central crop of the 112 um patch, e.g. uni2_h_c56 = central 56 um (112 px), resized to 224.
        um = int(name.split("_c")[-1])
        enc = build_encoder("uni2_h", device)
        enc.name, enc.center_crop_px = name, int(round(um / 0.5))
        return enc
    else:
        raise ValueError(name)
    enc.device = device
    enc.model.eval().to(device)
    for p in enc.model.parameters():
        p.requires_grad_(False)
    return enc


In [ ]:
%%writefile /content/lib/hestfm_data.py
"""On-disk format shared by the Colab notebooks (writers) and local analysis (readers).

    <root>/<dataset>/<encoder>/<sample>.h5      embeddings (N, D) float16, barcodes (N,), coords (N, 2)
    <root>/<dataset>/expression/<sample>.h5     barcodes (N,), genes (G,), coords (N, 2), and either
                                                counts (N, G) dense, or a CSR group `counts_csr`
                                                {data, indices, indptr, shape}; optional extra datasets.

`dataset` is e.g. "phase1/IDC" or "phase2/xenium5k". Rows are matched on barcodes, never on position.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import scipy.sparse as sp


def _str(a) -> np.ndarray:
    a = np.asarray(a).reshape(-1)
    return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in a])


@dataclass
class SampleData:
    sample_id: str
    X: np.ndarray            # (N, D) float32 embeddings
    counts: np.ndarray       # (N, G) float32 raw counts
    barcodes: np.ndarray
    coords: np.ndarray
    genes: np.ndarray


def read_embeddings(path: str | Path):
    with h5py.File(path, "r") as f:
        emb = f["embeddings"][:]
        bc = _str(f["barcodes"][:])
        coords = f["coords"][:] if "coords" in f else np.zeros((len(bc), 2))
    return emb, bc, coords


def read_expression(path: str | Path, genes: list[str] | None = None):
    with h5py.File(path, "r") as f:
        bc = _str(f["barcodes"][:])
        all_genes = _str(f["genes"][:])
        coords = f["coords"][:] if "coords" in f else np.zeros((len(bc), 2))
        if "counts_csr" in f:
            g = f["counts_csr"]
            M = sp.csr_matrix((g["data"][:].astype(np.float32), g["indices"][:].astype(np.int32),
                               g["indptr"][:].astype(np.int64)), shape=tuple(g["shape"][:]))
        else:
            M = f["counts"][:]
    if genes is not None:
        pos = {g: i for i, g in enumerate(all_genes)}
        missing = [g for g in genes if g not in pos]
        if missing:
            raise KeyError(f"{path}: genes missing: {missing[:5]}... ({len(missing)})")
        idx = np.array([pos[g] for g in genes])
        M = M[:, idx]
        all_genes = np.asarray(genes)
    return M, bc, coords, all_genes


def load_sample(root: str | Path, dataset: str, encoder: str, sample_id: str,
                genes: list[str] | None = None) -> SampleData:
    root = Path(root) / dataset
    emb, ebc, coords = read_embeddings(root / encoder / f"{sample_id}.h5")
    M, xbc, _, g = read_expression(root / "expression" / f"{sample_id}.h5", genes)
    pos = {b: i for i, b in enumerate(xbc)}
    keep = np.array([b in pos for b in ebc])
    if keep.sum() == 0:
        raise ValueError(f"{sample_id}: no barcode overlap between embeddings and expression")
    rows = np.array([pos[b] for b in ebc[keep]])
    Y = M[rows]
    Y = Y.toarray() if sp.issparse(Y) else np.asarray(Y)
    return SampleData(sample_id, emb[keep].astype(np.float32), Y.astype(np.float32), ebc[keep], coords[keep], g)


def stack(samples: list[SampleData]):
    """Concatenate samples; returns X, counts, sample_ids, barcodes, coords."""
    X = np.concatenate([s.X for s in samples])
    Y = np.concatenate([s.counts for s in samples])
    sid = np.concatenate([[s.sample_id] * len(s.X) for s in samples])
    bc = np.concatenate([s.barcodes for s in samples])
    coords = np.concatenate([s.coords for s in samples])
    return X, Y, sid, bc, coords


def write_embeddings(path, embeddings, barcodes, coords):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, "w") as f:
        f.create_dataset("embeddings", data=np.asarray(embeddings, dtype=np.float16))
        f.create_dataset("barcodes", data=np.asarray(barcodes).astype("S"))
        f.create_dataset("coords", data=np.asarray(coords))


def write_expression(path, counts, barcodes, genes, coords, **extra):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, "w") as f:
        if sp.issparse(counts):
            c = sp.csr_matrix(counts)
            c.sum_duplicates()
            data = c.data
            if np.allclose(data, np.round(data)) and (len(data) == 0 or data.max() < 65535):
                data = data.astype(np.uint16)   # integer counts: compact
            idx_dtype = np.int16 if c.shape[1] < 32767 else np.int32
            g = f.create_group("counts_csr")
            kw = dict(compression="gzip", compression_opts=4)
            g.create_dataset("data", data=data, **kw)
            g.create_dataset("indices", data=c.indices.astype(idx_dtype), **kw)
            g.create_dataset("indptr", data=c.indptr.astype(np.int64), **kw)
            g["shape"] = np.array(c.shape)
        else:
            f.create_dataset("counts", data=np.asarray(counts, dtype=np.float32))
        f.create_dataset("barcodes", data=np.asarray(barcodes).astype("S"))
        f.create_dataset("genes", data=np.asarray(genes).astype("S"))
        f.create_dataset("coords", data=np.asarray(coords))
        for k, v in extra.items():
            f.create_dataset(k, data=v)


In [ ]:
%%writefile /content/lib/pseudospots.py
"""Patch-aligned pseudo-spots from Xenium transcripts (inlined into the Phase 2 Colab notebook).

Definition (documented in PROGRESS.md / paper):
- A transcript counts if `is_gene`, `qv >= 20`, and it is assigned to a cell (`cell_id != "UNASSIGNED"`).
- A cell's position is the mean H&E position (`he_x`, `he_y`, full-resolution px) of its counted transcripts.
- A pseudo-spot is the square window of side `scale * S` centred on a HEST patch, where S is the patch footprint in
  full-res px (`patch_size_src`, 112 um at 0.5 um/px after resizing to 224 px). Its counts are the summed counts of
  all cells whose position falls inside the window. scale=1 is the primary, patch-aligned definition; 0.5 and 2 are
  used for the window-size sensitivity analysis. Windows of neighbouring patches overlap (HEST grid spacing 100 um).

Transcript files are streamed one row group at a time; nothing is held per transcript.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
import scipy.sparse as sp

UNASSIGNED = {"UNASSIGNED", "-1", "", "None", "nan"}
COLUMNS = ["cell_id", "feature_name", "he_x", "he_y", "qv", "is_gene"]


@dataclass
class CellMatrix:
    counts: sp.csr_matrix   # (n_cells, n_genes) int
    genes: np.ndarray
    centroids: np.ndarray   # (n_cells, 2) mean he_x, he_y
    n_transcripts: np.ndarray
    cell_hash: np.ndarray   # uint64 hash of cell_id


def _hash(strings) -> np.ndarray:
    return pd.util.hash_array(np.asarray(strings, dtype=object))


class TranscriptAggregator:
    """Streaming cell x gene counts + centroids."""

    def __init__(self, min_qv: float = 20.0):
        self.min_qv = min_qv
        self.cell_index = pd.Index(np.array([], dtype=np.uint64))
        self.gene_pos: dict[str, int] = {}
        self.sum_x = np.zeros(0)
        self.sum_y = np.zeros(0)
        self.n = np.zeros(0, dtype=np.int64)
        self.parts: list[tuple[np.ndarray, np.ndarray, np.ndarray]] = []
        self.n_seen = self.n_kept = 0

    def _grow(self, size):
        if size > len(self.n):
            new = max(size, int(len(self.n) * 1.5) + 1024)
            for k in ("sum_x", "sum_y"):
                a = np.zeros(new); a[: len(getattr(self, k))] = getattr(self, k); setattr(self, k, a)
            a = np.zeros(new, dtype=np.int64); a[: len(self.n)] = self.n; self.n = a

    def add(self, table) -> None:
        """`table`: pyarrow Table or pandas DataFrame with COLUMNS."""
        import pyarrow as pa
        import pyarrow.compute as pc
        if isinstance(table, pd.DataFrame):
            table = pa.Table.from_pandas(table, preserve_index=False)
        self.n_seen += table.num_rows
        cid = pc.cast(table["cell_id"], pa.string())
        mask = pc.and_(pc.and_(pc.cast(table["is_gene"], pa.bool_()), pc.greater_equal(table["qv"], self.min_qv)),
                       pc.invert(pc.is_in(cid, value_set=pa.array(sorted(UNASSIGNED)))))
        mask = pc.fill_null(mask, False)
        t = table.filter(mask)
        if t.num_rows == 0:
            return
        self.n_kept += t.num_rows
        # cells: dictionary-encode within the chunk, hash only the unique ids
        cd = pc.dictionary_encode(pc.cast(t["cell_id"], pa.string())).combine_chunks()
        h = _hash(cd.dictionary.to_numpy(zero_copy_only=False))
        pos = self.cell_index.get_indexer(h)
        new = pos < 0
        if new.any():
            start = len(self.cell_index)
            self.cell_index = self.cell_index.append(pd.Index(h[new]))
            pos[new] = np.arange(start, start + new.sum())
        cell = pos[cd.indices.to_numpy(zero_copy_only=False)].astype(np.int64)
        # genes
        gd = pc.dictionary_encode(pc.cast(t["feature_name"], pa.string())).combine_chunks()
        gnames = gd.dictionary.to_pylist()
        gmap = np.array([self.gene_pos.setdefault(g, len(self.gene_pos)) for g in gnames], dtype=np.int64)
        gene = gmap[gd.indices.to_numpy(zero_copy_only=False)]
        x = t["he_x"].to_numpy().astype(np.float64)
        y = t["he_y"].to_numpy().astype(np.float64)
        self._grow(len(self.cell_index))
        m = len(self.cell_index)
        self.sum_x[:m] += np.bincount(cell, weights=x, minlength=m)
        self.sum_y[:m] += np.bincount(cell, weights=y, minlength=m)
        self.n[:m] += np.bincount(cell, minlength=m)
        key = cell * (1 << 20) + gene
        u, c = np.unique(key, return_counts=True)
        self.parts.append(((u >> 20).astype(np.uint32), (u & ((1 << 20) - 1)).astype(np.uint32), c.astype(np.uint32)))

    def finalize(self) -> CellMatrix:
        n_cells, n_genes = len(self.cell_index), len(self.gene_pos)
        if self.parts:
            r = np.concatenate([p[0] for p in self.parts])
            c = np.concatenate([p[1] for p in self.parts])
            v = np.concatenate([p[2] for p in self.parts])
        else:
            r = c = v = np.zeros(0, dtype=np.uint32)
        self.parts = []
        M = sp.coo_matrix((v, (r, c)), shape=(n_cells, n_genes)).tocsr()
        M.sum_duplicates()
        names = np.array(sorted(self.gene_pos, key=self.gene_pos.get))
        order = np.argsort(names)
        n = self.n[:n_cells]
        cent = np.c_[self.sum_x[:n_cells] / np.maximum(n, 1), self.sum_y[:n_cells] / np.maximum(n, 1)]
        return CellMatrix(M[:, order].tocsr(), names[order], cent, n, self.cell_index.to_numpy())


def window_assignment(centroids: np.ndarray, patch_topleft: np.ndarray, patch_size_px: float,
                      scale: float = 1.0, swap_xy: bool = False) -> sp.csr_matrix:
    """(n_patches, n_cells) 0/1 matrix: cell centroid inside the square window of side scale*S at the patch centre."""
    from scipy.spatial import cKDTree
    c = centroids[:, ::-1] if swap_xy else centroids
    centers = np.asarray(patch_topleft, dtype=float) + patch_size_px / 2.0
    half = scale * patch_size_px / 2.0
    tree = cKDTree(c)
    lists = tree.query_ball_point(centers, r=half, p=np.inf)
    rows = np.repeat(np.arange(len(centers)), [len(l) for l in lists])
    cols = np.concatenate([np.asarray(l, dtype=np.int64) for l in lists]) if len(rows) else np.zeros(0, dtype=np.int64)
    # query_ball_point is inclusive (<= half); make the window half-open like a pixel footprint
    if len(cols):
        d = np.abs(c[cols] - centers[rows])
        ok = (d < half).all(1)
        rows, cols = rows[ok], cols[ok]
    return sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(centers), len(c)))


def orientation_check(centroids, patch_topleft, patch_size_px, max_cells=200_000, seed=0) -> dict:
    """Fraction of cells inside any patch footprint for (x, y) and swapped (y, x). The larger one is the right frame."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(centroids), size=min(max_cells, len(centroids)), replace=False)
    out = {}
    for swap in (False, True):
        A = window_assignment(centroids[idx], patch_topleft, patch_size_px, 1.0, swap)
        out["swapped" if swap else "xy"] = float((np.asarray(A.sum(0)).ravel() > 0).mean())
    return out


def pseudospot_counts(A: sp.csr_matrix, cells: CellMatrix) -> sp.csr_matrix:
    return (A @ cells.counts).tocsr().astype(np.float32)


In [ ]:
%%writefile /content/lib/celltypes.py
"""Coarse marker-based cell typing and gene cell-type specificity (inlined into the Phase 2 notebook).

This is deliberately simple: each cell gets the type whose marker set has the highest mean log-normalised
expression, if that score is above a floor; otherwise "unassigned". Cells with few transcripts are "low_quality".
It is used to (1) describe pseudo-spot composition and (2) compute per-gene specificity (tau) across types.
"""
from __future__ import annotations

import numpy as np
import scipy.sparse as sp

MARKERS = {
    "epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19", "CDH1", "KRT7"],
    "fibroblast": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRA"],
    "endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "KDR"],
    "smooth_muscle_pericyte": ["ACTA2", "MYH11", "TAGLN", "DES", "RGS5"],
    "t_nk": ["CD3E", "CD3D", "CD2", "CD8A", "CD247", "NKG7"],
    "b": ["MS4A1", "CD79A", "CD19", "CD22", "BANK1"],
    "plasma": ["MZB1", "JCHAIN", "TNFRSF17", "DERL3"],
    "myeloid": ["CD68", "CD163", "CD14", "LYZ", "CSF1R", "C1QA", "C1QB"],
    "mast": ["TPSAB1", "CPA3", "KIT", "MS4A2"],
}
MIN_TRANSCRIPTS = 20
SCORE_FLOOR = 0.3


def lognorm(counts: sp.csr_matrix, target: float = 100.0) -> sp.csr_matrix:
    tot = np.asarray(counts.sum(1)).ravel()
    scale = target / np.maximum(tot, 1)
    X = sp.diags(scale) @ counts.astype(np.float32)
    X = X.tocsr()
    X.data = np.log1p(X.data)
    return X


def annotate(counts: sp.csr_matrix, genes, markers=MARKERS, min_transcripts=MIN_TRANSCRIPTS, floor=SCORE_FLOOR):
    """Returns (labels array of str, type names, used markers dict)."""
    genes = list(genes)
    pos = {g: i for i, g in enumerate(genes)}
    X = lognorm(counts)
    types, scores, used = [], [], {}
    for t, ms in markers.items():
        idx = [pos[m] for m in ms if m in pos]
        if not idx:
            continue
        used[t] = [genes[i] for i in idx]
        types.append(t)
        scores.append(np.asarray(X[:, idx].mean(1)).ravel())
    S = np.vstack(scores).T
    best = S.argmax(1)
    labels = np.array(types, dtype=object)[best]
    labels[S.max(1) < floor] = "unassigned"
    labels[np.asarray(counts.sum(1)).ravel() < min_transcripts] = "low_quality"
    return labels.astype(str), types, used


def type_profiles(counts: sp.csr_matrix, labels, types):
    """(n_types, n_genes) mean log-normalised expression per type, and cells per type."""
    X = lognorm(counts)
    prof, n = [], []
    for t in types:
        m = labels == t
        n.append(int(m.sum()))
        prof.append(np.asarray(X[m].mean(0)).ravel() if m.any() else np.zeros(X.shape[1]))
    return np.vstack(prof), np.array(n)


def tau(profiles: np.ndarray) -> np.ndarray:
    """Tissue-specificity index (Yanai et al. 2005) across types: 0 = uniform, 1 = one type only. NaN if all zero."""
    P = np.asarray(profiles, dtype=float).T          # genes x types
    mx = P.max(1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        t = (1 - P / mx).sum(1) / (P.shape[1] - 1)
    t[mx.ravel() <= 0] = np.nan
    return t


def composition(A: sp.csr_matrix, labels, types) -> np.ndarray:
    """(n_spots, n_types + 2) cell counts per type in each pseudo-spot; last two columns unassigned, low_quality."""
    cols = list(types) + ["unassigned", "low_quality"]
    onehot = sp.csr_matrix((np.ones(len(labels)), (np.arange(len(labels)),
                            [cols.index(l) for l in labels])), shape=(len(labels), len(cols)))
    return np.asarray((A @ onehot).todense()), cols


In [ ]:
from encoders import build_encoder, embed
from hestfm_data import write_expression
from pseudospots import TranscriptAggregator, orientation_check, window_assignment, pseudospot_counts, COLUMNS
from celltypes import annotate, type_profiles, tau, composition

In [ ]:
import json, time, shutil, gc
import numpy as np, h5py, torch

def atomic_h5(path, write_fn):
    tmp = path + '.tmp'
    write_fn(tmp)
    os.replace(tmp, path)

def save_embeddings(path, emb, barcodes, coords):
    def w(p):
        with h5py.File(p, 'w') as f:
            f.create_dataset('embeddings', data=np.asarray(emb, dtype=np.float16))
            f.create_dataset('barcodes', data=np.asarray(barcodes).astype('S'))
            f.create_dataset('coords', data=np.asarray(coords))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    atomic_h5(path, w)

def read_patch_meta(h5_path):
    with h5py.File(h5_path, 'r') as f:
        bkey = 'barcode' if 'barcode' in f else 'barcodes'
        ikey = img_key(f)
        bc = f[bkey][:].reshape(-1)
        bc = np.array([b.decode() if isinstance(b, bytes) else str(b) for b in bc])
        return bc, f['coords'][:], dict(f[ikey].attrs), f[ikey].shape[0]

def img_key(f):
    return next(k for k in ('img', 'imgs', 'images') if k in f)

def embed_patch_file(h5_path, encoders, chunk=512):
    # One pass over the patch file; every encoder sees each chunk. Returns {name: (N, D) float16}.
    out = {e.name: [] for e in encoders}
    with h5py.File(h5_path, 'r') as f:
        imgs = f[img_key(f)]; n = imgs.shape[0]
        for i in range(0, n, chunk):
            batch = imgs[i:i + chunk]
            for e in encoders:
                out[e.name].append(embed(batch, e, amp_dtype=AMP))
    return {k: np.concatenate(v) for k, v in out.items()}

AMP = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print('autocast dtype:', AMP)

In [ ]:
import pandas as pd, pyarrow.parquet as pq, scipy.sparse as sp, anndata as ad, traceback
from huggingface_hub import hf_hub_download, HfFileSystem

def hest_file(rel):
    return hf_hub_download(HEST, rel, repo_type='dataset', local_dir=SCRATCH)

def qc_path(sid, root=OUT):
    return f'{root}/qc/{sid}.json'

def load_qc(sid, root=OUT):
    p = qc_path(sid, root)
    return json.load(open(p)) if os.path.exists(p) else {}

def save_qc(sid, qc, root=OUT):
    os.makedirs(f'{root}/qc', exist_ok=True)
    json.dump(qc, open(qc_path(sid, root) + '.tmp', 'w'), indent=2)
    os.replace(qc_path(sid, root) + '.tmp', qc_path(sid, root))

CONTROL_PREFIX = ('NegControl', 'BLANK', 'Unassigned', 'Deprecated', 'Intergenic', 'antisense', 'Genomic')

## Load encoders (H-optimus-1 is optional: if access is still pending it is skipped)

In [ ]:
skipped, loaded = {}, {}
for n in ENCODERS:
    try:
        if n.startswith('uni2_h_c') and 'uni2_h' in loaded:
            import copy
            e = copy.copy(loaded['uni2_h']); e.name = n; e.center_crop_px = int(round(int(n.split('_c')[-1]) / 0.5))
        else:
            e = build_encoder(n)
        loaded[n] = e
    except Exception as ex:
        skipped[n] = f'{type(ex).__name__}: {str(ex)[:200]}'
        print(f'SKIP encoder {n}: {skipped[n]}')
encoders = [loaded[n] for n in ENCODERS if n in loaded]
print('encoders:', [(e.name, e.dim) for e in encoders], '| skipped:', list(skipped))
print(torch.cuda.memory_allocated() / 1e9, 'GB on GPU')

## Section A: Xenium 5k

In [ ]:
def embed_sample(sid, encs, out_root, qc):
    todo = [e for e in encs if not os.path.exists(f'{out_root}/{e.name}/{sid}.h5')]
    if not todo:
        return
    t0 = time.time()
    p = hest_file(f'patches/{sid}.h5')
    bc, coords, attrs, n = read_patch_meta(p)
    embs = embed_patch_file(p, todo)
    for e in todo:
        save_embeddings(f'{out_root}/{e.name}/{sid}.h5', embs[e.name], bc, coords)
    qc.update({'n_patches': int(n), 'patch_size_src': float(attrs.get('patch_size_src', np.nan)),
               'pixel_size': float(attrs.get('pixel_size', np.nan))})
    qc.setdefault('embed_s', {})['+'.join(e.name for e in todo)] = round(time.time() - t0, 1)
    os.remove(p)
    print(f'  {sid}: embedded {n} patches with {[e.name for e in todo]} in {time.time() - t0:.0f}s')

def patch_meta_from_outputs(sid, qc):
    ref = next(e.name for e in encoders if os.path.exists(f'{OUT}/{e.name}/{sid}.h5'))
    with h5py.File(f'{OUT}/{ref}/{sid}.h5') as f:
        bc = f['barcodes'][:].astype(str); coords = f['coords'][:]
    return bc, coords, qc['patch_size_src']

def stream_cells(sid):
    # Download the whole transcript parquet to VM disk (much faster than remote row-group reads),
    # aggregate it row group by row group, then delete it before the next sample.
    t0 = time.time()
    local = hest_file(f'transcripts/{sid}_transcripts.parquet')
    print(f'    downloaded transcripts ({os.path.getsize(local) / 1e9:.1f} GB) in {(time.time() - t0) / 60:.1f} min')
    try:
        pf = pq.ParquetFile(local)
        n_rg = pf.metadata.num_row_groups
        agg = TranscriptAggregator(min_qv=MIN_QV)
        t1 = time.time()
        for i in range(n_rg):
            agg.add(pf.read_row_group(i, columns=COLUMNS))
            if i % 50 == 0 or i == n_rg - 1:
                el = time.time() - t1
                print(f'    row group {i + 1}/{n_rg}  kept {agg.n_kept / 1e6:.1f}M / seen {agg.n_seen / 1e6:.1f}M '
                      f'transcripts, {len(agg.cell_index)} cells, {el / 60:.1f} min (eta {el / (i + 1) * (n_rg - i - 1) / 60:.0f} min)')
        return agg.finalize(), agg
    finally:
        os.remove(local)

def pseudospots_sample(sid, qc):
    outs = {k: f'{OUT}/{k}/{sid}.h5' for k in WINDOWS}
    if all(os.path.exists(p) for p in outs.values()) and os.path.exists(f'{OUT}/cells/{sid}.h5'):
        return
    bc, coords, S = patch_meta_from_outputs(sid, qc)
    t0 = time.time()
    cells, agg = stream_cells(sid)
    keep = np.array([not g.startswith(CONTROL_PREFIX) for g in cells.genes])
    cells.counts, cells.genes = cells.counts[:, keep].tocsr(), cells.genes[keep]
    ori = orientation_check(cells.centroids, coords, S)
    swap = ori['swapped'] > ori['xy']
    qc.update({'transcripts_seen': int(agg.n_seen), 'transcripts_kept': int(agg.n_kept),
               'n_cells': int(cells.counts.shape[0]), 'n_genes': int(len(cells.genes)),
               'orientation_frac_inside': ori, 'swap_xy': bool(swap), 'stream_min': round((time.time() - t0) / 60, 1),
               'median_transcripts_per_cell': float(np.median(cells.n_transcripts))})
    if max(ori.values()) < 0.5:
        print(f'  WARNING {sid}: only {max(ori.values()):.2f} of cells fall inside patches; check alignment')
    labels, types, used = annotate(cells.counts, cells.genes)
    prof, n_per = type_profiles(cells.counts, labels, types)
    for key, scale in WINDOWS.items():
        A = window_assignment(cells.centroids, coords, S, scale, swap)
        X = pseudospot_counts(A, cells)
        extra = {'n_cells': np.asarray(A.sum(1)).ravel().astype(np.int32)}
        if scale == 1.0:
            comp, cols = composition(A, labels, types)
            extra.update({'composition': comp.astype(np.int32), 'composition_cols': np.array(cols).astype('S')})
        tmp = outs[key] + '.tmp'
        write_expression(tmp, X, bc, cells.genes, coords, **extra); os.replace(tmp, outs[key])
        qc.setdefault('windows', {})[key] = {'scale': scale, 'mean_cells': float(extra['n_cells'].mean()),
                                             'mean_counts': float(X.sum() / X.shape[0])}
    os.makedirs(f'{OUT}/cells', exist_ok=True)
    def w(p):
        with h5py.File(p, 'w') as f:
            f['type_profiles'] = prof.astype(np.float32); f['types'] = np.array(types).astype('S')
            f['n_per_type'] = n_per; f['genes'] = cells.genes.astype('S'); f['tau'] = tau(prof).astype(np.float32)
            f['gene_total_counts'] = np.asarray(cells.counts.sum(0)).ravel()
            f['gene_cells_detected'] = np.asarray((cells.counts > 0).sum(0)).ravel()
            f.attrs['markers'] = json.dumps(used)
    atomic_h5(f'{OUT}/cells/{sid}.h5', w)
    qc['cell_types'] = dict(zip(*np.unique(labels, return_counts=True)))
    qc['cell_types'] = {k: int(v) for k, v in qc['cell_types'].items()}
    del cells, agg; gc.collect()
    print(f'  {sid}: pseudo-spots done in {(time.time() - t0) / 60:.1f} min, swap_xy={swap}, orientation={ori}')

def hest_crosscheck(sid, qc):
    if 'hest_crosscheck' in qc:
        return
    p = hest_file(f'st/{sid}.h5ad')
    a = ad.read_h5ad(p)
    with h5py.File(f'{OUT}/expression/{sid}.h5') as f:
        bc = f['barcodes'][:].astype(str); genes = f['genes'][:].astype(str)
        g = f['counts_csr']; M = sp.csr_matrix((g['data'][:].astype(np.float32), g['indices'][:].astype(np.int32), g['indptr'][:]), shape=tuple(g['shape'][:]))
    vset, oset = set(a.var_names), set(a.obs_names)
    common = [x for x in genes if x in vset]
    okb = np.array([b in oset for b in bc])
    H = a[bc[okb], common].X
    H = H.toarray() if hasattr(H, 'toarray') else np.asarray(H)
    gpos = {x: i for i, x in enumerate(genes)}
    gi = [gpos[x] for x in common]
    O = M[okb][:, gi].toarray()
    tot_r = float(np.corrcoef(O.sum(1), H.sum(1))[0, 1])
    lo, lh = np.log1p(O), np.log1p(H)
    r = [np.corrcoef(lo[:, k], lh[:, k])[0, 1] for k in range(len(common)) if lo[:, k].std() > 0 and lh[:, k].std() > 0]
    qc['hest_crosscheck'] = {'n_spots': int(okb.sum()), 'n_genes': len(common), 'total_counts_pearson': round(tot_r, 4),
                             'per_gene_log1p_pearson_median': round(float(np.median(r)), 4),
                             'ratio_ours_to_hest_counts': round(float(O.sum() / max(H.sum(), 1)), 3)}
    os.remove(p)
    print(f'  {sid}: HEST cross-check {qc["hest_crosscheck"]}')

In [ ]:
t_all = time.time()
for sid in SAMPLES:
    qc = load_qc(sid)
    print(f'== {sid}')
    for step in (lambda: embed_sample(sid, encoders, OUT, qc), lambda: pseudospots_sample(sid, qc),
                 lambda: hest_crosscheck(sid, qc)):
        try:
            step()
        except Exception:
            traceback.print_exc()
            qc.setdefault('errors', []).append(traceback.format_exc()[-800:])
            save_qc(sid, qc)
            break
        save_qc(sid, qc)
    gc.collect(); torch.cuda.empty_cache()
print(f'Section A wall time {(time.time() - t_all) / 60:.1f} min')

## Section B (Phase 4): Visium lung / ovary / cervix, same encoders, Xenium 5k genes only

In [ ]:
genes_5k = set()
for sid in SAMPLES:
    p = f'{OUT}/expression/{sid}.h5'
    if os.path.exists(p):
        with h5py.File(p) as f:
            genes_5k |= set(f['genes'][:].astype(str))
print(len(genes_5k), 'Xenium 5k genes (union)')
vis_encoders = [e for e in encoders if e.name in VISIUM_ENCODERS]
for sid in VISIUM:
    qc = load_qc(sid, OUT_VIS)
    try:
        embed_sample(sid, vis_encoders, OUT_VIS, qc)
        xo = f'{OUT_VIS}/expression/{sid}.h5'
        if not os.path.exists(xo):
            with h5py.File(f'{OUT_VIS}/{vis_encoders[0].name}/{sid}.h5') as f:
                bc = f['barcodes'][:].astype(str); coords = f['coords'][:]
            a = ad.read_h5ad(hest_file(f'st/{sid}.h5ad'))
            a.var_names_make_unique()
            g = sorted(genes_5k & set(a.var_names))
            oset = set(a.obs_names)
            okb = np.array([b in oset for b in bc])
            X = a[bc[okb], g].X
            X = sp.csr_matrix(X) if not sp.issparse(X) else X.tocsr()
            tmp = xo + '.tmp'
            write_expression(tmp, X, bc[okb], np.array(g), coords[okb],
                             total_counts_all_genes=np.asarray(a[bc[okb]].X.sum(1)).ravel().astype(np.float32))
            os.replace(tmp, xo)
            qc.update({'n_spots_expr': int(okb.sum()), 'n_genes_5k': len(g)})
            os.remove(f'{SCRATCH}/st/{sid}.h5ad')
        print(f'  {sid}: ok {qc.get("n_spots_expr")}')
    except Exception:
        traceback.print_exc(); qc.setdefault('errors', []).append(traceback.format_exc()[-800:])
    save_qc(sid, qc, OUT_VIS)

## Summary

In [ ]:
rows = []
for sid in SAMPLES:
    q = load_qc(sid)
    rows.append({'sample': sid, 'patches': q.get('n_patches'), 'cells': q.get('n_cells'), 'genes': q.get('n_genes'),
                 'swap_xy': q.get('swap_xy'), 'inside_frac': max(q.get('orientation_frac_inside', {'x': np.nan}).values()),
                 'mean_cells_per_spot': q.get('windows', {}).get('expression', {}).get('mean_cells'),
                 'hest_total_r': q.get('hest_crosscheck', {}).get('total_counts_pearson'),
                 'hest_gene_r_median': q.get('hest_crosscheck', {}).get('per_gene_log1p_pearson_median'),
                 'errors': len(q.get('errors', []))})
summary = pd.DataFrame(rows); print(summary.to_string())
vis = pd.DataFrame([{'sample': s, **{k: load_qc(s, OUT_VIS).get(k) for k in ('n_patches', 'n_spots_expr', 'n_genes_5k')},
                     'errors': len(load_qc(s, OUT_VIS).get('errors', []))} for s in VISIUM]); print(vis.to_string())
json.dump({'xenium5k': rows, 'visium': vis.to_dict('records'), 'skipped_encoders': skipped,
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'},
          open(f'{OUT}/run_summary.json', 'w'), indent=2, default=str)

In [ ]:
!du -sh {OUT}/* {OUT_VIS}